In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load your CSV
df = pd.read_csv("basic_shape_features.csv")

# Separate metadata and features
non_features = df[['filename', 'country']]
features = df.drop(columns= non_features)

# Standardize features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

features_scaled = pd.DataFrame(features_scaled, columns=features.columns)

In [ ]:
from scipy.spatial.distance import mahalanobis
from numpy.linalg import inv

X = features_scaled.values
cov_matrix = np.cov(X, rowvar=False)
inv_cov = inv(cov_matrix)
mean_distr = X.mean(axis=0)

distances = []

for i in range(len(X)):
    dist = mahalanobis(X[i], mean_distr, inv_cov)
    distances.append(dist)

df['mahal_dist'] = distances

# Chi-square threshold
from scipy.stats import chi2
threshold = chi2.ppf((1 - 0.001), df=X.shape[1])

df_clean = df[df['mahal_dist'] < np.sqrt(threshold)]

In [ ]:
features_cols = [
    'area_px', 'perimeter_px', 'major_axis_px', 'minor_axis_px',
    'aspect_ratio', 'extent', 'roundness',
    'circularity', 'convexity', 'solidity','orientation']

df_clean[features_cols].describe()

In [ ]:
import matplotlib.pyplot as plt

df_clean[features_cols].hist(bins=30, figsize=(12,10))
plt.tight_layout()
plt.show()

In [ ]:
df_clean['log_area'] = np.log(df_clean['area_px'])
df_clean['log_perimeter'] = np.log(df_clean['perimeter_px'])
df_clean['log_major_axis'] = np.log(df_clean['major_axis_px'])
df_clean['log_minor_axis'] = np.log(df_clean['minor_axis_px'])

In [ ]:
log_cols = [
    'log_area', 'log_perimeter', 'log_major_axis', 'log_minor_axis',
    'aspect_ratio', 'extent', 'roundness',
    'circularity', 'convexity', 'solidity']

In [ ]:
scaler = StandardScaler()
features_rescaled = scaler.fit_transform(df_clean[log_cols])

features_rescaled = pd.DataFrame(features_rescaled, columns=df_clean[log_cols].columns)

In [ ]:
#caclulating PCA

from sklearn.decomposition import PCA

pca = PCA()
pca_scores = pca.fit_transform(features_rescaled)

explained = pca.explained_variance_ratio_

print(explained[:6])

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(log_cols))],
    index=log_cols
)

print(loadings[['PC1','PC2','PC3']])

In [ ]:
#PCA visualizations

pc_df = pd.DataFrame(pca_scores[:, :3], columns=['PC1','PC2','PC3'])
pc_df['country'] = df_clean['country'].values

plt.figure(figsize=(8,6))

for c in pc_df['country'].unique():
    subset = pc_df[pc_df['country'] == c]
    plt.scatter(subset['PC1'], subset['PC2'], label=c, alpha=0.6)

plt.xlabel('PC1 (46.1%)')
plt.ylabel('PC2 (32.3%)')
plt.legend()
plt.show()

In [ ]:
#centroid analysis
centroids = pc_df.groupby('country')[['PC1','PC2','PC3']].mean()
print(centroids)

In [ ]:
from scipy.spatial.distance import pdist, squareform

centroid_distances = pd.DataFrame(
    squareform(pdist(centroids)),
    index=centroids.index,
    columns=centroids.index
)

print(centroid_distances)

In [ ]:
#FIGURE 1 — Distribution Plots 
features_log = [
    'log_area',
    'log_perimeter',
    'log_major_axis',
    'log_minor_axis',
    'aspect_ratio',
    'circularity'
]

plt.figure(figsize=(12, 8))

for i, col in enumerate(features_log, 1):
    plt.subplot(2, 3, i)
    plt.hist(df_clean[col], bins=30)
    plt.title(col)
    plt.xlabel(col)
    plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
#FIGURE 2 — Correlation Heatmap
import seaborn as sns
import numpy as np

shape_features = [
    'log_area',
    'log_perimeter',
    'log_major_axis',
    'log_minor_axis',
    'aspect_ratio',
    'extent',
    'roundness',
    'circularity',
    'convexity',
    'solidity'
]

corr_matrix = df_clean[shape_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix,
            annot=True,
            cmap='coolwarm',
            fmt=".2f",
            square=True)

plt.title("Correlation Matrix of Basic Shape Features")
plt.show()

In [ ]:
#FIGURE 3 — Scree Plot
plt.figure(figsize=(8, 5))

plt.plot(range(1, len(pca.explained_variance_ratio_) + 1),
         pca.explained_variance_ratio_ * 100,
         marker='o')

plt.xlabel("Principal Component")
plt.ylabel("Explained Variance (%)")
plt.title("Scree Plot")
plt.show()

In [ ]:
X = df_clean[shape_features]
X_scaled = scaler.fit_transform(X)
X_pca = pca.fit_transform(X_scaled)

In [ ]:
#FIGURE 4 — PCA Scatter Plot (PC1 vs PC2)
pca_df = pd.DataFrame(X_pca[:, :3], columns=['PC1', 'PC2', 'PC3'])
pca_df['country'] = df_clean['country'].values

plt.figure(figsize=(8, 6))

for country in pca_df['country'].unique():
    subset = pca_df[pca_df['country'] == country]
    plt.scatter(subset['PC1'], subset['PC2'], label=country)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Scatter Plot (PC1 vs PC2)")
plt.legend()
plt.show()

In [ ]:
#table 1 descriptive features
desc_table = df_clean[shape_features].describe().T
desc_table = desc_table[['mean', 'std', 'min', 'max']].round(2)
desc_table.to_csv('desc_table.csv', index=True)

In [ ]:
#table 2 PCA variance
variance_table = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
    "Variance (%)": pca.explained_variance_ratio_ * 100,
    "Cumulative (%)": np.cumsum(pca.explained_variance_ratio_) * 100
}).round(3)

variance_table.to_csv('variance_table.csv', index=True)

In [ ]:
# table 3 PCA loadings
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(len(shape_features))],
    index=shape_features
).round(3)

loadings = loadings[['PC1', 'PC2', 'PC3']]
loadings.to_csv('loadings_table.csv', index=True)

In [ ]:
from scipy.spatial.distance import pdist, squareform
centroids = pca_df.groupby('country')[['PC1', 'PC2', 'PC3']].mean()
distance_matrix = pd.DataFrame(
    squareform(pdist(centroids)),
    index=centroids.index,
    columns=centroids.index
)
#centroids

distance_matrix